# LLM Agora Demo
Interactively run a simple Agora session with configurable agents, LLM backends, and turn limits.

## Instructions
- Ensure `.env` defines `OPENROUTER_API_KEY`.
- Tweak `agent_configs` / `turns_per_agent` (and snapshot flags) to explore different models, prompts, and persistence paths.
- Run cells sequentially to execute Agora sessions, inspect histories, and optionally save/load snapshots.


In [ ]:
import sys
sys.path.append("../src")

from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from agora.agora import Agora
from agora.agent import Agent, build_system_prompt
from agora.llm import OpenRouterClient
from agora.persistence import load_snapshot, save_snapshot


# Purely public debate

- Simplest possible configuration

In [ ]:
# --- Configuration ---
turns_per_agent = 2
agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-5-mini',
        'system_prompt': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
        'response_instruction': 'Alpha, offer your next public utterance.',
    },
    {
        'name': 'Beta',
        'model': 'openai/gpt-5-mini',
        'system_prompt': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
        'response_instruction': 'Beta, offer your next public utterance.',
    },
]


In [ ]:
# --- Build agents and run the Agora ---
client = OpenRouterClient()
try:
    agents = []
    for cfg in agent_configs:
        system_prompt = build_system_prompt(cfg, total_agents=len(agent_configs))
        agent = Agent(
            name=cfg['name'],
            model=cfg['model'],
            llm_client=client,
            system_prompt=system_prompt,
            response_instruction=cfg['response_instruction'],
        )
        agents.append(agent)

    agora = Agora(agents)
    history = agora.run(max_turns_per_agent=turns_per_agent, verbose=True)
finally:
    client.close()


In [ ]:
# --- Inspect each agent's perspective on the history ---
for agent in agents:
    print(f"\n### History visible to {agent.name}")
    for turn in agent.view_history():
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")


# Public debate with private reflections
- 3 agents example
- The option to save and/or load the state is also available

In [ ]:
# --- Private reflection configuration ---
private_turns_per_agent = 2
private_agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-5-mini',
        'self_role': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Beta', 'role': 'Beta is your potential buyer.'},
            {'name': 'Gamma', 'role': 'Gamma is in the audience.'}
        ],
        'response_instruction': 'Alpha, offer your next public utterance.',
        'private_response_instruction': 'Alpha, what do you think of the interaction so far?',
    },
    {
        'name': 'Beta',
        'model': 'openai/gpt-5-mini',
        'self_role': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Alpha', 'role': 'Alpha is your salesman.'},
            {'name': 'Gamma', 'role': 'Gamma is in the audience.'}
        ],
        'response_instruction': 'Beta, offer your next public utterance.',
        'private_response_instruction': 'Beta, what do you think of the interaction so far?',
    },
    {
        'name': 'Gamma',
        'model': 'openai/gpt-5-mini',
        'self_role': 'You are Gamma, listening to a conversation between a salesman and a potential buyer. You helpfully ad-lib their conversation in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Alpha', 'role': 'Alpha is the salesman.'},
            {'name': 'Beta', 'role': 'Beta is the potential buyer.'}
        ],
        'response_instruction': 'Gamma, offer your next public utterance.',
        'private_response_instruction': 'Gamma, what do you think of the interaction so far?',
    },
]

snapshot_path = Path('../snapshots/reflection_snapshot.json')
load_snapshot_flag = True  # Set True to resume from a saved snapshot (ignores private_agent_configs)
save_snapshot_flag = False  # Set True to persist the new session.


In [ ]:
# --- Run Agora with private reflections ---
private_agents = []
private_client = OpenRouterClient()
try:
    if load_snapshot_flag and snapshot_path.exists():
        def _snapshot_factory(state):
            return private_client
        private_agora = load_snapshot(snapshot_path, _snapshot_factory)
        private_agents = list(private_agora.agents)
        print(f'Loaded snapshot from {snapshot_path}')
    else:
        for cfg in private_agent_configs:
            system_prompt = build_system_prompt(cfg, total_agents=len(private_agent_configs))
            agent = Agent(
                name=cfg['name'],
                model=cfg['model'],
                llm_client=private_client,
                system_prompt=system_prompt,
                response_instruction=cfg['response_instruction'],
                private_response_instruction=cfg.get('private_response_instruction'),
            )
            private_agents.append(agent)
        private_agora = Agora(private_agents)

    private_history = private_agora.run(max_turns_per_agent=private_turns_per_agent, verbose=True)
finally:
    private_client.close()


In [ ]:
# --- Save snapshot (optional) ---
if save_snapshot_flag:
    save_snapshot(snapshot_path, private_agora)
    print(f'Snapshot saved to {snapshot_path}')
else:
    print('Snapshot not saved (set save_snapshot_flag=True to persist).')


In [ ]:
# --- Inspect each agent's perspective on the history ---
for agent in private_agents:
    print(f"\n### Full history visible to {agent.name}")
    for turn in agent.view_history():
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        if turn.role == 'reflection':
            print(f"Turn {turn.turn_id:02d} | {speaker} (private): {turn.private_reflection}")
        else:
            print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")
